In [ ]:
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 


pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome', 'enso']

predict_n = 36
max_epiweek = 16
    
boxcox = False

TEST_YEAR = 2023


In [2]:
batch_size = 2
epochs = 250
media =True
verbose = 1
doenca = 'chikungunya'
min_delta = 0.0
patience= 25
lr = 2e-4
min_year = 2015
model_name = f'enso_media_transf'

In [3]:
df = prep.load_cases_data(filename= f'./data/{doenca}.csv.gz')

enso = prep.load_enso_weekly()

enso_neutro = prep.load_enso_weekly(filename='data/enso_weekly_neutro.csv')

In [ ]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    X_train, y_train, X_future, norm_values, norm_enso =  prep.generate_regional_train_samples(df_reg, enso, 
                                                                                    TEST_YEAR,
                                                                                    max_epiweek = max_epiweek,
                                                                                    columns_to_normalize = columns_to_normalize,
                                                                                    min_year = min_year, 
                                                                                    boxcox = boxcox,
                                                                                    media = media)

    model = mc.LSTMWithFutureCovariatesV2(

                    hidden=64,

                    past_features=5,

                    future_cov_size=predict_n,

                    predict_n=predict_n,

                    dropout=0.2
                        )
    
    #model_path = f'./saved_models/trained_dengue_{region}_{TEST_YEAR-1}_enso_media.pt'
    #model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    #model.to(device)  
    
    trained_model, hist = mc.train_model(

            model=model,

            X_train=X_train,

            X_future=X_future,

            Y_train=y_train,

            label=label,

            batch_size=batch_size,

            epochs=epochs,

            lr=lr,

            patience=patience,

            verbose=verbose,

            doenca=doenca
)


Sul
Epoch 1/250 | Train: 0.0460 | Val: 0.0325
Epoch 2/250 | Train: 0.0460 | Val: 0.0307
Epoch 3/250 | Train: 0.0422 | Val: 0.0303
Epoch 4/250 | Train: 0.0430 | Val: 0.0301
Epoch 5/250 | Train: 0.0414 | Val: 0.0300
Epoch 6/250 | Train: 0.0409 | Val: 0.0299
Epoch 7/250 | Train: 0.0396 | Val: 0.0302
Epoch 8/250 | Train: 0.0381 | Val: 0.0300
Epoch 9/250 | Train: 0.0386 | Val: 0.0301
Epoch 10/250 | Train: 0.0388 | Val: 0.0304
Epoch 11/250 | Train: 0.0379 | Val: 0.0306
Epoch 12/250 | Train: 0.0379 | Val: 0.0304
Epoch 13/250 | Train: 0.0353 | Val: 0.0305
Epoch 14/250 | Train: 0.0362 | Val: 0.0305
Epoch 15/250 | Train: 0.0345 | Val: 0.0306
Epoch 16/250 | Train: 0.0371 | Val: 0.0308
Epoch 17/250 | Train: 0.0358 | Val: 0.0308
Epoch 18/250 | Train: 0.0372 | Val: 0.0308
Epoch 19/250 | Train: 0.0347 | Val: 0.0309
Epoch 20/250 | Train: 0.0352 | Val: 0.0311
Epoch 21/250 | Train: 0.0356 | Val: 0.0311
Epoch 22/250 | Train: 0.0361 | Val: 0.0312
Epoch 23/250 | Train: 0.0357 | Val: 0.0314
Epoch 24/250 | T

"\n    for state in regioes_estados[region]:\n\n        print(state)\n\n        df_state = df_reg.loc[df_reg.uf == state]\n\n        df_preds =  mc.sum_regions_predictions(trained_model, df_state, enso, TEST_YEAR, columns_to_normalize,\n                                    max_epiweek = max_epiweek,\n                                    boxcox = boxcox,\n                                    n_passes = 500,\n                                    min_year= min_year, \n                                    media = media)\n\n        df_preds.to_csv(f'predictions/preds_{doenca}_{state}_{TEST_YEAR}_forecast_{model_name}.csv')\n\n        df_preds_neutro =  mc.sum_regions_predictions(trained_model, df_state, enso_neutro, TEST_YEAR, columns_to_normalize,\n                                    max_epiweek = max_epiweek,\n                                    boxcox = boxcox,\n                                    n_passes = 500,\n                                    min_year= min_year, \n                     